# Image → Text → Summary → Sentiment using `langchain_openai`

## Objective

In this notebook we build a complete multimodal NLP workflow using **LangChain + OpenAI**.

The input is an image containing motivational text. We will:

1. Load and display the image.
2. Convert the image to Base64 so it can be sent to an OpenAI multimodal model.
3. Use `langchain_openai.ChatOpenAI` to **extract all visible text from the image**.
4. Send the extracted text to another LangChain prompt for **summarization**.
5. Send the same extracted text to another prompt for **sentiment analysis**.
6. Combine the results into a small final report.

### Flow

```text
Image
  ↓
Base64 encoding
  ↓
ChatOpenAI multimodal model
  ↓
Extracted text
  ├─────────────→ Summarization prompt → Summary
  │
  └─────────────→ Sentiment prompt     → Sentiment
```

This notebook does **not** use Hugging Face or Transformers.

## Step 1 — Install the required libraries

`langchain-openai` provides the LangChain integration for OpenAI models.

`python-dotenv` is used to safely load the API key from a `.env` file.

> Run the installation cell once. If the libraries are already installed, you can skip it.

In [ ]:
# Uncomment and run once if required.
# %pip install -U langchain-openai langchain-core python-dotenv pillow pandas

## Step 2 — Import libraries

We need:

- `Path` to work with file paths.
- `base64` to convert the image into text-safe Base64 data.
- `PIL.Image` and `IPython.display` to display the image.
- `ChatOpenAI` to call an OpenAI multimodal model.
- `ChatPromptTemplate` to build reusable LangChain prompts.

In [ ]:
import os
import base64
from pathlib import Path

import pandas as pd
from PIL import Image
from IPython.display import display

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

## Step 3 — Load the OpenAI API key

Create a file named `.env` in the same folder as this notebook:

```text
OPENAI_API_KEY=your_openai_api_key_here
```

Keeping the API key outside the notebook is safer than writing it directly in a code cell.

In [ ]:
load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    print("OPENAI_API_KEY loaded successfully.")
else:
    print("OPENAI_API_KEY is not set. Add it to a .env file before calling the model.")

## Step 4 — Load and display the image

The notebook expects the generated image file:

```text
believe_in_yourself.png
```

Displaying the image first is useful because it lets us visually confirm what the model is expected to read.

In [ ]:
IMAGE_PATH = Path("believe_in_yourself.png")

image = Image.open(IMAGE_PATH)
print("Image size:", image.size)
display(image)

## Step 5 — Convert the image to Base64

OpenAI's multimodal API can receive an image as inline Base64 data.

Base64 converts the binary image bytes into text characters that can safely be included in the request.

The conversion does **not** extract the text. It only prepares the image for transmission to the model.

In [ ]:
def image_to_base64(image_path: Path) -> str:
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

image_base64 = image_to_base64(IMAGE_PATH)

print("Base64 characters:", len(image_base64))
print("Preview:", image_base64[:80] + "...")

## Step 6 — Create the multimodal OpenAI model

`ChatOpenAI` is the LangChain wrapper around OpenAI chat models.

We use a model that supports image input. `temperature=0` is useful for extraction tasks because we want the model to be deterministic and avoid creative rewriting.

> If this model name is unavailable in your account in the future, replace it with a currently available OpenAI model that supports image input.

In [ ]:
VISION_MODEL = "gpt-4.1-mini"

vision_llm = ChatOpenAI(
    model=VISION_MODEL,
    temperature=0
)

print("Vision model:", VISION_MODEL)

## Step 7 — Extract the text from the image

Here the image and the instruction are sent together.

Notice that the human message contains **two content blocks**:

1. a text instruction;
2. the Base64 image.

The prompt tells the model to perform OCR-like extraction and preserve the original wording and order.

In [ ]:
ocr_message = {
    "role": "user",
    "content": [
        {
            "type": "text",
            "text": (
                "Read this image carefully and extract all visible text. "
                "Preserve the original reading order. "
                "Do not summarize, explain, or add any new words. "
                "Return only the extracted text."
            ),
        },
        {
            "type": "image",
            "base64": image_base64,
            "mime_type": "image/png",
        },
    ],
}

ocr_response = vision_llm.invoke([ocr_message])
extracted_text = ocr_response.content

print("EXTRACTED TEXT")
print("=" * 70)
print(extracted_text)

## Step 8 — Understand the extracted text

At this point the image-processing stage is complete.

The next two tasks operate on **text**, not directly on pixels:

```text
Extracted text
     |
     +----> Summarization
     |
     +----> Sentiment analysis
```

This separation is useful in real applications because the extracted text can also be saved, searched, classified, translated, embedded, or stored in a database.

## Step 9 — Create a summarization chain

A LangChain chain can be created by connecting:

```text
Prompt Template | LLM
```

The prompt asks the model to reduce the extracted motivational content into a short summary while retaining the main message.

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a concise text summarizer. "
        "Summarize the supplied text accurately without adding information."
    ),
    (
        "human",
        "Summarize the following extracted image text in 2-3 sentences:\n\n{text}"
    )
])

summary_chain = summary_prompt | vision_llm

## Step 10 — Generate the summary

In [ ]:
summary_response = summary_chain.invoke({"text": extracted_text})
summary = summary_response.content

print("SUMMARY")
print("=" * 70)
print(summary)

## Step 11 — Create a sentiment-analysis chain

Sentiment analysis identifies the emotional orientation of text.

For this simple example we use three labels:

- **Positive**
- **Negative**
- **Neutral**

The prompt also asks for a short reason so students can understand why the label was selected.

In [ ]:
sentiment_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a sentiment classifier. "
        "Classify text as Positive, Negative, or Neutral."
    ),
    (
        "human",
        "Analyze this text:\n\n{text}\n\n"
        "Return exactly this format:\n"
        "Sentiment: <Positive/Negative/Neutral>\n"
        "Reason: <one short sentence>"
    )
])

sentiment_chain = sentiment_prompt | vision_llm

## Step 12 — Run sentiment analysis

In [ ]:
sentiment_response = sentiment_chain.invoke({"text": extracted_text})
sentiment_result = sentiment_response.content

print("SENTIMENT ANALYSIS")
print("=" * 70)
print(sentiment_result)

## Step 13 — Create a final report

We now combine the three outputs:

1. extracted text;
2. summary;
3. sentiment result.

This is useful when the results need to be displayed in an application or saved to CSV/Excel/database storage.

In [ ]:
report = pd.DataFrame([{
    "image": IMAGE_PATH.name,
    "extracted_text": extracted_text,
    "summary": summary,
    "sentiment_analysis": sentiment_result
}])

report

## Step 14 — Save the output

The final report can be exported to CSV.

In [ ]:
OUTPUT_FILE = "openai_image_text_analysis.csv"
report.to_csv(OUTPUT_FILE, index=False)
print("Saved:", OUTPUT_FILE)

## What happened behind the scenes?

### Image extraction
The multimodal model receives both the image pixels and the instruction. It interprets the visual content and generates the visible text.

### Summarization
The extracted text is tokenized internally by the OpenAI service. The model identifies the most important ideas and generates a shorter representation.

### Sentiment analysis
The model interprets the wording and emotional tone, then maps the text to the requested sentiment labels.

### Important distinction
This workflow is **not traditional OCR + separate ML classifiers**. A multimodal generative model performs the image understanding, and the LLM performs the downstream language tasks.